# Skill-Centric Grounded Reasoning — single instance (LLM via OpenRouter)

This notebook runs **one end-to-end pass** through the pipeline using **Qwen** (via OpenRouter) as the reasoning engine:

Question + retrieved corpus context + skill contracts → **Qwen LLM call** → structured answer with provenance.

The skill contracts (`skills/*/SKILL.md`) are injected into the system prompt so the model knows the expected output shape, units conventions, and arithmetic rules.

**Corpus:** repo `corpus/` mirrors competition layout (`index.txt` + `treasury_bulletin_YYYY_MM.txt`). In Docker that maps to `/app/corpus/`.

**Harness:** uses the `opencode` harness with `openrouter` model backend (see `arena.yaml`).

## 0. Paths, imports, and OpenRouter config

In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
from pathlib import Path

import httpx

# Repo root = parent of notebooks/
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

CORPUS_ROOT = REPO_ROOT / "corpus"
SKILLS_ROOT = REPO_ROOT / "skills"
INDEX_PATH = CORPUS_ROOT / "index.txt"

assert INDEX_PATH.is_file(), f"Missing {INDEX_PATH} — run from repo root or set REPO_ROOT"

# OpenRouter config
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
QWEN_MODEL = "qwen/qwen3-235b-a22b"

assert OPENROUTER_API_KEY, (
    "Set OPENROUTER_API_KEY env var before running. "
    "See arena.yaml for the expected env layout."
)

print("CORPUS_ROOT:", CORPUS_ROOT.resolve())
print("Model:", QWEN_MODEL)

## 1. Instance: question

In [ ]:
QUESTION = (
    "In the January 1953 Treasury bulletin excerpt, what is the sum of the Outstanding column "
    "(in millions of dollars) for categories Alpha and Beta?"
)

print(QUESTION)

## 2. Retrieve corpus documents

Light deterministic retrieval: match date tokens from the question against `index.txt` filenames, then read the full text of matching bulletins.

In [ ]:
MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}


def extract_date_tokens(q: str) -> tuple[int | None, int | None]:
    """Pull year and month from the question text."""
    ql = q.lower()
    year_m = re.search(r"\b(19\d{2}|20\d{2})\b", q)
    year = int(year_m.group(1)) if year_m else None
    month = None
    for name, num in MONTHS.items():
        if name in ql:
            month = num
            break
    return year, month


def retrieve_documents(question: str, corpus: Path) -> list[dict]:
    """Return list of {filename, content} for bulletins matching the question's date."""
    year, month = extract_date_tokens(question)
    index_entries = [
        line.strip()
        for line in (corpus / "index.txt").read_text().splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]
    matched = []
    for entry in index_entries:
        m = re.match(r"treasury_bulletin_(\d{4})_(\d{2})\.txt$", entry)
        if not m:
            continue
        fy, fm = int(m.group(1)), int(m.group(2))
        if (year is None or fy == year) and (month is None or fm == month):
            p = corpus / entry
            if p.is_file():
                matched.append({"filename": entry, "content": p.read_text()})
    # Fallback: if no date match, return all documents
    if not matched:
        for entry in index_entries:
            p = corpus / entry
            if p.is_file():
                matched.append({"filename": entry, "content": p.read_text()})
    return matched


documents = retrieve_documents(QUESTION, CORPUS_ROOT)
for doc in documents:
    print(f"Retrieved: {doc['filename']} ({len(doc['content'])} chars)")

## 3. Load skill contracts for the system prompt

All four `SKILL.md` files are concatenated and injected into the system message so the LLM knows the expected reasoning procedure, output schema, and edge-case rules.

In [ ]:
SKILL_NAMES = [
    "bulletin-retriever",
    "plaintext-table-parser",
    "arithmetic-verifier",
    "cross-doc-aggregator",
]


def load_all_skills(skills_root: Path) -> str:
    """Concatenate all SKILL.md files into one block for the system prompt."""
    parts = []
    for name in SKILL_NAMES:
        p = skills_root / name / "SKILL.md"
        if p.is_file():
            parts.append(p.read_text().strip())
    return "\n\n---\n\n".join(parts)


skills_text = load_all_skills(SKILLS_ROOT)
print(f"Loaded {len(SKILL_NAMES)} skill contracts ({len(skills_text)} chars)")
print(skills_text[:300], "\n...")

## 4. Build the LLM prompt

The system prompt embeds the skill contracts. The user message contains the question and the full text of every retrieved document.

In [ ]:
SYSTEM_PROMPT = f"""\
You are a grounded-reasoning agent for Treasury bulletin analysis.

You have the following skill contracts that describe your capabilities and the
procedures you must follow. Adhere to them strictly:

{skills_text}

IMPORTANT RULES:
- Parentheses around a number mean NEGATIVE, e.g. (450) = -450.
- Do NOT double-count "Total" or subtotal rows; use only individual category rows.
- Use exact decimal arithmetic. Show your calculation trace step by step.
- Units must come from the document caption/header (e.g. "Millions of dollars").
- Your final answer MUST be valid JSON matching this schema:

{{
  "answer": {{
    "value": "<numeric result as string>",
    "unit": "<unit from document>",
    "narrative": "<one-sentence plain-English answer>"
  }},
  "provenance": {{
    "source_files": ["<filename(s)>"],
    "evidence_rows": [
      {{"label": "<row label>", "raw_value": "<as printed>", "parsed_value": "<numeric>"}}
    ],
    "calculation_trace": [
      {{"op": "<operation>", "args": ["<values>"], "result": "<result>"}}
    ],
    "unit_source": "<where the unit was found in the document>"
  }}
}}

Return ONLY the JSON object, no markdown fences, no commentary outside the JSON.
"""


def build_user_message(question: str, docs: list[dict]) -> str:
    parts = [f"QUESTION: {question}\n"]
    for doc in docs:
        parts.append(f"--- DOCUMENT: {doc['filename']} ---")
        parts.append(doc["content"])
        parts.append("--- END DOCUMENT ---\n")
    parts.append("Answer the question using ONLY the documents above. Return JSON.")
    return "\n".join(parts)


user_message = build_user_message(QUESTION, documents)
print("System prompt length:", len(SYSTEM_PROMPT))
print("User message length:", len(user_message))
print("\n--- User message preview ---")
print(user_message[:500])

## 5. Call Qwen via OpenRouter

Single synchronous HTTP call using `httpx`. The response is parsed as JSON.

In [ ]:
def call_qwen(system: str, user: str, model: str = QWEN_MODEL) -> dict:
    """Send a chat completion request to OpenRouter and return the parsed JSON answer."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.0,
        "max_tokens": 2048,
    }
    resp = httpx.post(OPENROUTER_BASE_URL, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    raw_content = data["choices"][0]["message"]["content"]
    # Strip markdown fences if the model wraps them anyway
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw_content.strip())
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return json.loads(cleaned)


result = call_qwen(SYSTEM_PROMPT, user_message)
print(json.dumps(result, indent=2))

## 6. Validate the LLM response

Check that the answer matches expected value and the provenance is well-formed.

In [ ]:
def validate_result(result: dict) -> None:
    """Run basic assertions on the LLM's structured output."""
    answer = result.get("answer", {})
    prov = result.get("provenance", {})

    # Value check — Alpha=1250, Beta=-450, sum=800
    val = answer.get("value", "")
    assert val == "800", f"Expected value '800', got '{val}'"

    # Unit check
    unit = answer.get("unit", "").lower().replace(" ", "_")
    assert "million" in unit, f"Expected unit containing 'million', got '{answer.get('unit')}'"

    # Provenance must reference source file
    assert prov.get("source_files"), "Missing source_files in provenance"

    # Evidence rows should include Alpha and Beta
    labels = {r.get("label") for r in prov.get("evidence_rows", [])}
    assert "Alpha" in labels and "Beta" in labels, f"Evidence rows missing Alpha/Beta, got {labels}"

    # Calculation trace should exist
    assert prov.get("calculation_trace"), "Missing calculation_trace"

    print("All validations passed.")


validate_result(result)
print("\nFinal answer:")
print(json.dumps(result["answer"], indent=2))

## 7. One-shot runner (full pipeline)

Convenience wrapper that runs retrieval → LLM call → validation in one call.

In [ ]:
def run_pipeline(question: str, corpus: Path, skills_root: Path) -> dict:
    """Full pipeline: retrieve docs → build prompt → call Qwen → return structured result."""
    docs = retrieve_documents(question, corpus)
    skills = load_all_skills(skills_root)

    system = f"""\
You are a grounded-reasoning agent for Treasury bulletin analysis.

You have the following skill contracts that describe your capabilities and the
procedures you must follow. Adhere to them strictly:

{skills}

IMPORTANT RULES:
- Parentheses around a number mean NEGATIVE, e.g. (450) = -450.
- Do NOT double-count "Total" or subtotal rows; use only individual category rows.
- Use exact decimal arithmetic. Show your calculation trace step by step.
- Units must come from the document caption/header (e.g. "Millions of dollars").
- Your final answer MUST be valid JSON matching this schema:

{{
  "answer": {{
    "value": "<numeric result as string>",
    "unit": "<unit from document>",
    "narrative": "<one-sentence plain-English answer>"
  }},
  "provenance": {{
    "source_files": ["<filename(s)>"],
    "evidence_rows": [
      {{"label": "<row label>", "raw_value": "<as printed>", "parsed_value": "<numeric>"}}
    ],
    "calculation_trace": [
      {{"op": "<operation>", "args": ["<values>"], "result": "<result>"}}
    ],
    "unit_source": "<where the unit was found in the document>"
  }}
}}

Return ONLY the JSON object, no markdown fences, no commentary outside the JSON.
"""
    user = build_user_message(question, docs)
    return call_qwen(system, user)


full = run_pipeline(QUESTION, CORPUS_ROOT, SKILLS_ROOT)
validate_result(full)
print("\nOK — pipeline instance complete.")
full["answer"]